In [1]:
import polars as pl

In [2]:
df = pl.read_parquet("final_patient_data.parquet")

In [3]:
whole_df = df
non_icu_df = df.filter(pl.col("icu_stay_flag") == 0)
icu_df = df.filter(pl.col("icu_stay_flag") == 1)

In [9]:
disease_cols = [
    "Cardiopathie ischémique",
    "Fibrillation atriale",
    "Insuffisance cardiaque chronique",
    "Pacemaker",
    "Pontage aorto-coronarien",
    "Antécédent d'AVC"
]
drug_cols = [
    "beta_blocker",
    "ACEI",
    "ARB",
    "anti_aldosterone"
]

binary_cols = disease_cols + drug_cols

In [16]:
def summarize_binary_variable(data, col):
    """
    Return n (%) for one binary variable in one dataset.
    """
    total = data.height
    
    n = data.filter(pl.col(col) == 1).height
    
    percent = n / total * 100 if total > 0 else 0
    
    return f"{n} ({percent:.1f}%)"


def make_icu_table(df, binary_cols):
    """
    Create descriptive table with Whole, Non-ICU, and ICU columns.
    """
    
    whole_df = df
    non_icu_df = df.filter(pl.col("icu_stay_flag") == 0)
    icu_df = df.filter(pl.col("icu_stay_flag") == 1)
    
    rows = []
    
    # First row: number of patients
    rows.append({
        "Variable": "Number of patients",
        "Whole": str(whole_df.height),
        "Non-ICU": str(non_icu_df.height),
        "ICU": str(icu_df.height)
    })
    
    # Rows for diseases and treatments
    for col in binary_cols:
        rows.append({
            "Variable": col,
            "Whole": summarize_binary_variable(whole_df, col),
            "Non-ICU": summarize_binary_variable(non_icu_df, col),
            "ICU": summarize_binary_variable(icu_df, col)
        })
    
    return pl.DataFrame(rows)


icu_table = make_icu_table(df, binary_cols)
pl.Config.set_tbl_rows(-1)
icu_table

Variable,Whole,Non-ICU,ICU
str,str,str,str
"""Number of patients""","""10""","""8""","""2"""
"""Cardiopathie ischémique""","""2 (20.0%)""","""1 (12.5%)""","""1 (50.0%)"""
"""Fibrillation atriale""","""2 (20.0%)""","""2 (25.0%)""","""0 (0.0%)"""
"""Insuffisance cardiaque chroniq…","""1 (10.0%)""","""1 (12.5%)""","""0 (0.0%)"""
"""Pacemaker""","""0 (0.0%)""","""0 (0.0%)""","""0 (0.0%)"""
"""Pontage aorto-coronarien""","""0 (0.0%)""","""0 (0.0%)""","""0 (0.0%)"""
"""Antécédent d'AVC""","""1 (10.0%)""","""0 (0.0%)""","""1 (50.0%)"""
"""beta_blocker""","""3 (30.0%)""","""2 (25.0%)""","""1 (50.0%)"""
"""ACEI""","""1 (10.0%)""","""1 (12.5%)""","""0 (0.0%)"""


In [11]:
def summarize_continuous_variable(data, col):
    """
    Return median [Q1-Q3] for one continuous variable.
    """
    q = data.select([
        pl.col(col).median().alias("median"),
        pl.col(col).quantile(0.25).alias("q1"),
        pl.col(col).quantile(0.75).alias("q3")
    ])
    
    median = q["median"][0]
    q1 = q["q1"][0]
    q3 = q["q3"][0]
    
    return f"{median:.1f} [{q1:.1f}-{q3:.1f}]"

In [15]:
continuous_cols = [
    "age_at_admission",
    "Charlson Comorbidity Score",
    "HFRS Score"
]


def make_full_icu_table(df, continuous_cols, binary_cols):
    """
    Create descriptive table with continuous and binary variables.
    """
    
    whole_df = df
    non_icu_df = df.filter(pl.col("icu_stay_flag") == 0)
    icu_df = df.filter(pl.col("icu_stay_flag") == 1)
    
    rows = []
    
    # Number of patients
    rows.append({
        "Variable": "Number of patients",
        "Whole": str(whole_df.height),
        "Non-ICU": str(non_icu_df.height),
        "ICU": str(icu_df.height)
    })
    
    # Continuous variables
    for col in continuous_cols:
        rows.append({
            "Variable": col,
            "Whole": summarize_continuous_variable(whole_df, col),
            "Non-ICU": summarize_continuous_variable(non_icu_df, col),
            "ICU": summarize_continuous_variable(icu_df, col)
        })
    
    # Binary variables
    for col in binary_cols:
        rows.append({
            "Variable": col,
            "Whole": summarize_binary_variable(whole_df, col),
            "Non-ICU": summarize_binary_variable(non_icu_df, col),
            "ICU": summarize_binary_variable(icu_df, col)
        })
    
    return pl.DataFrame(rows)


icu_table = make_full_icu_table(df, continuous_cols, binary_cols)
pl.Config.set_tbl_rows(-1)
icu_table

Variable,Whole,Non-ICU,ICU
str,str,str,str
"""Number of patients""","""10""","""8""","""2"""
"""age_at_admission""","""82.5 [81.0-83.0]""","""83.0 [81.0-83.0]""","""81.0 [80.0-82.0]"""
"""Charlson Comorbidity Score""","""0.5 [0.0-1.0]""","""0.0 [0.0-1.0]""","""1.0 [1.0-1.0]"""
"""HFRS Score""","""0.0 [0.0-0.0]""","""0.0 [0.0-0.0]""","""1.9 [0.0-3.7]"""
"""Cardiopathie ischémique""","""2 (20.0%)""","""1 (12.5%)""","""1 (50.0%)"""
"""Fibrillation atriale""","""2 (20.0%)""","""2 (25.0%)""","""0 (0.0%)"""
"""Insuffisance cardiaque chroniq…","""1 (10.0%)""","""1 (12.5%)""","""0 (0.0%)"""
"""Pacemaker""","""0 (0.0%)""","""0 (0.0%)""","""0 (0.0%)"""
"""Pontage aorto-coronarien""","""0 (0.0%)""","""0 (0.0%)""","""0 (0.0%)"""
